# Delta Ops — VACUUM

Run **VACUUM … DRY RUN** then **VACUUM** on `fact_sales_fragmented` (from notebook 01). Show **DESCRIBE HISTORY** before and after.

In [ ]:
import sys
import importlib

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = notebook_path.split("/notebooks/")[0]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import src.delta_ops.vacuum_ops as vacuum_module
importlib.reload(vacuum_module)

from src.delta_ops.vacuum_ops import VacuumConfig, run_vacuum_demo
from src.ingestion.idempotent_loader import save_run_report_to_volume

config = VacuumConfig(retain_hours=168)
print("Loaded from:", vacuum_module.__file__)

In [ ]:
report = run_vacuum_demo(spark, config)
display(spark.sql(f"DESCRIBE HISTORY {config.target_table} LIMIT 10"))

In [ ]:
import json

print(json.dumps(report, indent=2, default=str))
try:
    save_run_report_to_volume(
        report,
        dbutils,
        "/Volumes/globalmart/metadata/run_reports/delta_vacuum.json",
    )
except Exception as exc:
    print("Volume save skipped:", exc)